In [ ]:
import requests
import json

url = "https://opensearch.pads.fim.uni-passau.de/msmarco-v2.1-segmented/_search"
auth = ("hana", "4DBpreroRMc!fPPczxaJ")
query = "Restaurants in Passau"

params = {
    "query": {"match": {"segment": query}},
    "size": 40,  # top-40
    "_source": ["docid", "segment"]  # Extract docid + text
}

resp = requests.get(url, auth=auth, json=params)
hits = resp.json()["hits"]["hits"]
scores = [h["_score"] for h in hits]
docids = [h["_source"]["docid"] for h in hits]

In [7]:
import urllib.request
import os

url = "https://msmarco.z22.web.core.windows.net/msmarcoranking/queries.tar.gz"
out_path = os.path.expanduser("~/queries.tar.gz")

print("Downloading queries.tar.gz (~30MB)...")
urllib.request.urlretrieve(url, out_path)
print("✓ Downloaded")

# Extract it
import tarfile
with tarfile.open(out_path, 'r:gz') as tar:
    tar.extractall(os.path.expanduser("~/msmarco/"))
    print("✓ Extracted:", tar.getnames())

✓ Downloaded


C:\Users\hanaz\AppData\Local\Temp\ipykernel_10624\3677070985.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(os.path.expanduser("~/msmarco/"))


✓ Extracted: ['queries.dev.tsv', 'queries.eval.tsv', 'queries.train.tsv']


In [8]:
import pandas as pd
from collections import defaultdict
import ir_datasets

# Read query texts directly — explicit UTF-8, no ir_datasets encoding issue
queries_tsv = os.path.expanduser("~/msmarco/queries.train.tsv")
query_text_map = {}
with open(queries_tsv, 'r', encoding='utf-8', errors='replace') as f:
    for line in f:
        parts = line.rstrip('\n').split('\t', 1)
        if len(parts) == 2:
            query_text_map[parts[0]] = parts[1]
print(f"✓ Loaded {len(query_text_map)} query texts")

# qrels only — safe, no encoding issues
dataset = ir_datasets.load("msmarco-passage/train/judged")
qrels_by_query = defaultdict(list)
for qrel in dataset.qrels_iter():
    qrels_by_query[qrel.query_id].append(qrel)
print(f"✓ Loaded {len(qrels_by_query)} queries with qrels")

queries_with_3plus = {
    qid: qrels for qid, qrels in qrels_by_query.items() if len(qrels) >= 3
}

csv_data = []
for query_id, qrels in queries_with_3plus.items():
    query_text = query_text_map.get(str(query_id), "")
    for qrel in qrels:
        csv_data.append({
            'query_id':   str(qrel.query_id),
            'query_text': query_text,
            'doc_id':     str(qrel.doc_id),
            'relevance':  int(qrel.relevance),
            'iteration':  str(getattr(qrel, 'iteration', '0'))
        })

df = pd.DataFrame(csv_data)
df.to_csv('msmarco_queries_3plus_qrels.csv', index=False, encoding='utf-8')
print(f"✓ Saved {len(csv_data)} rows → 'msmarco_queries_3plus_qrels.csv'")
print(df.groupby('query_id').size().describe())

✓ Loaded 808731 query texts
✓ Loaded 502939 queries with qrels
✓ Saved 11445 rows → 'msmarco_queries_3plus_qrels.csv'
count    3491.000000
mean        3.278430
std         0.584265
min         3.000000
25%         3.000000
50%         3.000000
75%         3.000000
max         7.000000
dtype: float64
